In [1]:
import pandas as pd
import numpy as np

train_model = pd.read_csv("../data/processed/champion_train_pd.csv")
test_model = pd.read_csv("../data/processed/champion_test_pd.csv")

print("Train shape:", train_model.shape)
print("Test shape :", test_model.shape)

Train shape: (70000, 10)
Test shape : (30000, 11)


In [2]:
print("Train columns:")
print(train_model.columns.tolist())

print("\nTest columns:")
print(test_model.columns.tolist())

Train columns:
['Credit_History_Years_WOE', 'Credit_Utilization_WOE', 'Debt_to_Income_WOE', 'Employment_Length_Years_WOE', 'Monthly_Income_WOE', 'Number_of_Delinquencies_30_WOE', 'Number_of_Delinquencies_60_WOE', 'Number_of_Delinquencies_90_WOE', 'Bad', 'PD']

Test columns:
['Credit_History_Years_WOE', 'Credit_Utilization_WOE', 'Debt_to_Income_WOE', 'Employment_Length_Years_WOE', 'Monthly_Income_WOE', 'Number_of_Delinquencies_30_WOE', 'Number_of_Delinquencies_60_WOE', 'Number_of_Delinquencies_90_WOE', 'Bad', 'Risk_Band', 'Score']


In [3]:
import joblib

model = joblib.load("../outputs/models/champion_logistic_model.pkl")

woe_features = [
    "Credit_History_Years_WOE",
    "Credit_Utilization_WOE",
    "Debt_to_Income_WOE",
    "Employment_Length_Years_WOE",
    "Monthly_Income_WOE",
    "Number_of_Delinquencies_30_WOE",
    "Number_of_Delinquencies_60_WOE",
    "Number_of_Delinquencies_90_WOE"
]

test_model["PD"] = model.predict_proba(
    test_model[woe_features]
)[:, 1]

print(test_model[["Bad", "PD"]].head())

   Bad        PD
0    0  0.053388
1    0  0.068223
2    0  0.078715
3    0  0.057718
4    0  0.094873


In [5]:
def calculate_psi(expected, actual, bins=10):
    breakpoints = np.quantile(
        expected,
        np.linspace(0, 1, bins + 1)
    )

    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_bins = pd.cut(expected, bins=breakpoints)
    actual_bins = pd.cut(actual, bins=breakpoints)

    expected_dist = (
        pd.Series(expected_bins)
        .value_counts(normalize=True, sort=False)
    )

    actual_dist = (
        pd.Series(actual_bins)
        .value_counts(normalize=True, sort=False)
    )

    expected_dist = expected_dist.replace(0, 0.0001)
    actual_dist = actual_dist.replace(0, 0.0001)

    psi = (
        (actual_dist - expected_dist)
        * np.log(actual_dist / expected_dist)
    ).sum()

    return psi


psi = calculate_psi(
    train_model["PD"].values,
    test_model["PD"].values
)

print("PD PSI:", round(psi, 4))

PD PSI: 0.0007


In [6]:
train_bad_rate = train_model["Bad"].mean()
test_bad_rate = test_model["Bad"].mean()

print("Train Bad Rate:", round(train_bad_rate, 4))
print("Test Bad Rate :", round(test_bad_rate, 4))
print("Difference    :", round(test_bad_rate - train_bad_rate, 4))

Train Bad Rate: 0.0682
Test Bad Rate : 0.0682
Difference    : 0.0


In [8]:
monitoring_summary = pd.DataFrame({
    "Metric": [
        "PD PSI",
        "Train Bad Rate",
        "Test Bad Rate",
        "Bad Rate Difference"
    ],
    "Value": [
        psi,
        train_bad_rate,
        test_bad_rate,
        test_bad_rate - train_bad_rate
    ]
})

monitoring_summary["Value"] = monitoring_summary["Value"].round(4)

monitoring_summary

,Metric,Value
0,PD PSI,0.0007
1,Train Bad Rate,0.0682
2,Test Bad Rate,0.0682
3,Bad Rate Difference,0.0000


In [9]:
monitoring_summary.to_csv(
    "../outputs/metrics/monitoring_summary.csv",
    index=False
)

print("Monitoring summary saved.")

Monitoring summary saved.


In [11]:
from sklearn.metrics import roc_auc_score, roc_curve

train_auc = roc_auc_score(
    train_model["Bad"],
    train_model["PD"]
)

test_auc = roc_auc_score(
    test_model["Bad"],
    test_model["PD"]
)

train_fpr, train_tpr, _ = roc_curve(
    train_model["Bad"],
    train_model["PD"]
)

test_fpr, test_tpr, _ = roc_curve(
    test_model["Bad"],
    test_model["PD"]
)

train_ks = np.max(train_tpr - train_fpr)
test_ks = np.max(test_tpr - test_fpr)

print("Train AUC :", round(train_auc, 4))
print("Test AUC  :", round(test_auc, 4))
print("Train KS  :", round(train_ks, 4))
print("Test KS   :", round(test_ks, 4))

Train AUC : 0.5987
Test AUC  : 0.5969
Train KS  : 0.144
Test KS   : 0.1455


In [12]:
def calculate_csi(expected, actual, bins=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    breakpoints = np.quantile(
        expected,
        np.linspace(0, 1, bins + 1)
    )

    # Remove duplicate bin edges
    breakpoints = np.unique(breakpoints)

    if len(breakpoints) < 3:
        return 0.0

    breakpoints[0] = -np.inf
    breakpoints[-1] = np.inf

    expected_bins = pd.cut(expected, bins=breakpoints)
    actual_bins = pd.cut(actual, bins=breakpoints)

    expected_dist = pd.Series(
        expected_bins
    ).value_counts(normalize=True, sort=False)

    actual_dist = pd.Series(
        actual_bins
    ).value_counts(normalize=True, sort=False)

    # Align bins
    actual_dist = actual_dist.reindex(
        expected_dist.index,
        fill_value=0
    )

    expected_dist = expected_dist.replace(0, 0.0001)
    actual_dist = actual_dist.replace(0, 0.0001)

    csi = (
        (actual_dist - expected_dist)
        * np.log(actual_dist / expected_dist)
    ).sum()

    return csi


csi_results = []

for feature in woe_features:
    csi_value = calculate_csi(
        train_model[feature],
        test_model[feature]
    )

    csi_results.append({
        "Feature": feature,
        "CSI": round(csi_value, 4)
    })

csi_results = pd.DataFrame(csi_results)

csi_results

,Feature,CSI
0,Credit_History_Years_WOE,0.0004
1,Credit_Utilization_WOE,0.0003
2,Debt_to_Income_WOE,0.0003
3,Employment_Length_Years_WOE,0.0002
4,Monthly_Income_WOE,0.0001
5,Number_of_Delinquencies_30_WOE,0.0000
6,Number_of_Delinquencies_60_WOE,0.0000
7,Number_of_Delinquencies_90_WOE,0.0000


In [13]:
csi_results.to_csv(
    "../outputs/metrics/csi_results.csv",
    index=False
)

print("CSI results saved.")

CSI results saved.
